In [ ]:
# This preprocessing is for the Telco dataset.
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def find_project_root():
    current_directory = Path.cwd().resolve()
    for candidate in (current_directory, *current_directory.parents):
        if (candidate / ".env").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root containing .env and data/")


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")


class PreprocessingTelco:
    categorical_features = [
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
    ]
    continuous_features = [
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "AvgMonthlyCharges",
    ]
    random_state = 42

    def __init__(self, dataset_path=None):
        self.dataset_path = self.resolve_dataset_path(dataset_path)
        self.telco_churn_df = pd.read_csv(self.dataset_path)

    def resolve_dataset_path(self, dataset_path=None):
        configured_path = (
            dataset_path
            or os.getenv("PATH_DATASET_TELCO", "").strip()
            or "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
        )
        configured_path = Path(configured_path).expanduser()

        if configured_path.is_absolute():
            candidates = [configured_path]
        else:
            candidates = [PROJECT_ROOT / configured_path]

        candidates.append(PROJECT_ROOT / "data" / configured_path.name)

        for candidate in candidates:
            if candidate.is_file():
                return candidate.resolve()

        raise FileNotFoundError(
            f"Telco dataset not found. Checked: {', '.join(map(str, candidates))}"
        )

    def preprocess(self):
        """Clean, engineer, split, encode, scale, and display the Telco data.

        No augmentation is applied: synthetic or image-style transformations are
        not appropriate for these customer-level tabular records.
        """

        columns = [
            "PhoneService",
            "MultipleLines",
            "InternetService",
            "OnlineSecurity",
            "OnlineBackup",
            "DeviceProtection",
            "TechSupport",
            "StreamingTV",
            "StreamingMovies",
            "MonthlyCharges",
            "TotalCharges",
            "Churn",
        ]

        prueba = self.telco_churn_df[columns]

        display(pd.DataFrame(prueba).head())

        self.select_features()
        self.drop_nan()
        self.make_feature_engineering()
        return self.split_dataset()

    def select_features(self):
        columns = [
            "gender",
            "SeniorCitizen",
            "Partner",
            "Dependents",
            "tenure",
            "PhoneService",
            "MultipleLines",
            "InternetService",
            "OnlineSecurity",
            "OnlineBackup",
            "DeviceProtection",
            "TechSupport",
            "StreamingTV",
            "StreamingMovies",
            "Contract",
            "PaperlessBilling",
            "PaymentMethod",
            "MonthlyCharges",
            "TotalCharges",
            "Churn",
        ]
        self.telco_churn_df = self.telco_churn_df[columns].copy()

    def encode(self, X_train, X_validation, X_test):
        """Fit categorical encoding on training data and transform all splits."""
        self.encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float64,
        )
        encoded_train = self.encoder.fit_transform(
            X_train[self.categorical_features]
        )
        encoded_validation = self.encoder.transform(
            X_validation[self.categorical_features]
        )
        encoded_test = self.encoder.transform(X_test[self.categorical_features])
        encoded_feature_names = self.encoder.get_feature_names_out(
            self.categorical_features
        )

        self.scaler = StandardScaler()
        self.scaler.fit(X_train[self.continuous_features])

        def transform_split(features, encoded_features):
            numeric_features = features.drop(
                columns=self.categorical_features
            ).copy()
            numeric_features[self.continuous_features] = numeric_features[
                self.continuous_features
            ].astype(float)
            numeric_features.loc[:, self.continuous_features] = (
                self.scaler.transform(features[self.continuous_features])
            )
            categorical_features = pd.DataFrame(
                encoded_features,
                columns=encoded_feature_names,
                index=features.index,
            )
            return pd.concat([numeric_features, categorical_features], axis=1)

        encoded_splits = (
            transform_split(X_train, encoded_train),
            transform_split(X_validation, encoded_validation),
            transform_split(X_test, encoded_test),
        )
        self.feature_names = encoded_splits[0].columns.tolist()
        return tuple(split.to_numpy() for split in encoded_splits)

    def drop_nan(self):
        """Impute blank TotalCharges for zero-tenure customers; drop other invalid rows."""
        self.telco_churn_df["TotalCharges"] = pd.to_numeric(
            self.telco_churn_df["TotalCharges"], errors="coerce"
        )
        zero_tenure_missing_charges = (
            self.telco_churn_df["TotalCharges"].isna()
            & self.telco_churn_df["tenure"].eq(0)
        )
        self.total_charges_imputed = int(zero_tenure_missing_charges.sum())
        self.telco_churn_df.loc[zero_tenure_missing_charges, "TotalCharges"] = 0

        self.telco_churn_df["Churn"] = self.telco_churn_df["Churn"].map(
            {"Yes": 1, "No": 0}
        )
        rows_before_cleaning = len(self.telco_churn_df)
        self.telco_churn_df = self.telco_churn_df.dropna().copy()
        self.rows_dropped = rows_before_cleaning - len(self.telco_churn_df)

    def make_feature_engineering(self):
        self.telco_churn_df["NewCustomer"] = (
            self.telco_churn_df["tenure"].eq(0).astype(int)
        )
        self.telco_churn_df["AvgMonthlyCharges"] = np.where(
            self.telco_churn_df["tenure"] > 0,
            self.telco_churn_df["TotalCharges"] / self.telco_churn_df["tenure"],
            0,
        )

    def split_dataset(self):
        """Create stratified 70/15/15 splits and fit transforms on train only.

        Validation data is exposed as ``X_validation`` and ``y_validation``.
        """
        X = self.telco_churn_df.drop(columns="Churn")
        y = self.telco_churn_df["Churn"].astype(int)

        X_train_raw, X_remaining, y_train, y_remaining = train_test_split(
            X,
            y,
            test_size=0.30,
            random_state=self.random_state,
            stratify=y,
        )
        X_validation_raw, X_test_raw, y_validation, y_test = train_test_split(
            X_remaining,
            y_remaining,
            test_size=0.50,
            random_state=self.random_state,
            stratify=y_remaining,
        )

        X_train, X_validation, X_test = self.encode(
            X_train_raw,
            X_validation_raw,
            X_test_raw,
        )
        self.X_validation = X_validation
        self.y_validation = y_validation.to_numpy()

        display(pd.DataFrame(X_train, columns=self.feature_names).head())

        self.split_sizes = {
            "train": len(y_train),
            "validation": len(y_validation),
            "test": len(y_test),
        }
        return X_train, X_test, y_train, y_test

In [9]:
preprocessing = PreprocessingTelco()
preprocessed_data = preprocessing.preprocess()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,NewCustomer,AvgMonthlyCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0.0,-1.114728,0.504286,-0.836504,0.0,0.393297,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,1.0,-1.195884,0.724189,-0.907742,0.0,0.288061,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.0,-1.195884,0.337292,-0.909574,0.0,0.242020,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,0.0,1.117066,0.515860,1.110332,0.0,0.522676,1.0,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,-0.830682,1.122660,-0.515129,0.0,0.937974,1.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
